In [33]:
import os
import logging
from langchain_community.document_loaders import PyPDFLoader

logging.getLogger("pypdf").setLevel(logging.ERROR)


def load_all_pdfs(folder_path: str) -> list:
    """
    Loads all PDF files from a folder and returns a combined list of Documents.
    Each page of each PDF becomes a separate Document, enriched with extra metadata.
    """
    all_docs = []

    pdf_files = [f for f in os.listdir(folder_path) if f.lower().endswith(".pdf")]

    print(f"Found {len(pdf_files)} PDF files in '{folder_path}' folder.")

    for filename in pdf_files:
        file_path = os.path.join(folder_path, filename)
        try:
            loader = PyPDFLoader(file_path)
            docs = loader.load()

            # Enrich metadata for each page/document FIRST
            for doc in docs:
                doc.metadata["filename"] = filename
                doc.metadata["total_pages"] = len(docs)
                doc.metadata["page_number"] = doc.metadata.get("page", 0) + 1

            # THEN print diagnostics for every page (no slicing = all pages)
            for doc in docs:
                print(f"File: {doc.metadata['filename']}, Page: {doc.metadata['page_number']}")
                print(f"Content length: {len(doc.page_content)}")
                print(f"Preview: {repr(doc.page_content[:200])}")
                print("---")

            all_docs.extend(docs)
            print(f"Loaded: {filename} ({len(docs)} pages)")
        except Exception as e:
            print(f"Failed to load {filename}: {e}")

    print(f"\nTotal documents (pages) loaded: {len(all_docs)}")
    return all_docs


all_docs = load_all_pdfs("../data")

print(f"\nTotal pages across all PDFs: {len(all_docs)}")

# Safely check metadata for a sample document (index 200 may not exist if you have fewer pages)
if len(all_docs) > 200:
    print(all_docs[200].metadata)
else:
    print(f"Only {len(all_docs)} total pages loaded — showing first one instead:")
    print(all_docs[0].metadata)

Found 17 PDF files in '../data' folder.
File: lec11Notes.pdf, Page: 1
Content length: 2687
Preview: "LEC-11: Normalisation \n1. Normalisation is a step towards DB optimisation.\n2. Functional Dependency (FD)\n1. It's a relationship between the primary key a ttribute (usually) of the relation to that of "
---
File: lec11Notes.pdf, Page: 2
Content length: 703
Preview: '2. 2NF \n1. Relation must be in 1NF. \n2. There should not be any partial dependency. \n1. All non-prime attributes must be fully dependent on PK. \n2. Non prime attribute can not depend on the part of th'
---
Loaded: lec11Notes.pdf (2 pages)
File: lec12Notes.pdf, Page: 1
Content length: 2025
Preview: 'LEC-12: Transaction \n1. Transaction\n1. A unit of work done against the DB in a logical sequence.\n2. Sequence is very important in transaction.\n3. It is a logical unit of work that contains one or more'
---
File: lec12Notes.pdf, Page: 2
Content length: 661
Preview: '1. When updates are made permanent on the DB. Then the T

In [36]:
### Text splitting and get into chunks

from langchain_text_splitters import RecursiveCharacterTextSplitter
from collections import defaultdict

# Initialize the splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)

# Split all_docs (from your PDF loader) into chunks
chunks = text_splitter.split_documents(all_docs)

print(f"Total chunks created: {len(chunks)}")

# Preview a sample chunk
print(chunks[0].page_content)
print(chunks[0].metadata)

# Group chunks by filename and print them all
chunks_by_file = defaultdict(list)
for chunk in chunks:
    chunks_by_file[chunk.metadata["filename"]].append(chunk)

for filename, file_chunks in chunks_by_file.items():
    print(f"\n{'='*80}")
    print(f"FILE: {filename} — {len(file_chunks)} chunks")
    print(f"{'='*80}")
    for i, chunk in enumerate(file_chunks):
        print(f"\n--- Chunk {i+1}/{len(file_chunks)} (Page {chunk.metadata.get('page_number')}) ---")
        print(chunk.page_content)
        print(f"[Length: {len(chunk.page_content)} chars]")

Total chunks created: 115
LEC-11: Normalisation 
1. Normalisation is a step towards DB optimisation.
2. Functional Dependency (FD)
1. It's a relationship between the primary key a ttribute (usually) of the relation to that of the other attribute of the
relation.
2. X -> Y, the left side of FD is known as a Determinant, the right side of the production is known as a Dependent.
3. Types of FD
1. Trivial FD
1. A → B has trivial functional dependency if B is a subset of A. A->A, B->B are also Trivial FD.
2. Non-trivial FD
1. A → B has a non-trivial functional dependency if B is not a subset of A. [A intersection B is NULL].
4. Rules of FD (Armstrong’s axioms)
1. Reflexive
1. If ‘A’ is a set of attributes and ‘B’ is a subset of ‘A’. Then, A→ B holds.
2. If A ⊇ B then A → B.
2. Augmentation
1. If B can be determined from A, then adding an attribute to this functional dependency won’t change
anything.
2. If A→ B holds, then AX→ BX holds too. ‘X’ being a set of attributes.
3. Transitivity
{'pr

In [37]:
### embedding and vector db
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity


d:\webProjects\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [41]:

class EmbeddingManager:
    """Handles document embedding generation using sentence-transformers and stores them in a vector database (ChromaDB)."""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            # Use the newer method name if available, fall back if not
            try:
                dim = self.model.get_embedding_dimension()
            except AttributeError:
                dim = self.model.get_sentence_embedding_dimension()
            print(f"Model loaded successfully. Embedding dimensions: {dim}")
        except Exception as e:
            print(f"Failed to load embedding model '{self.model_name}': {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("Embedding model is not loaded.")
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Embeddings generated. Shape: {embeddings.shape}")
        return embeddings


# Initialize the embedding manager (OUTSIDE the class, at top level)
embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 413.43it/s]


Model loaded successfully. Embedding dimensions: 384


In [43]:
### Vector Store Management

class VectorStore:
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self._initialize_store()

    def _initialize_store(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Create or get the collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings"}
            )

            print(f"Vector store initialized. Collection: '{self.collection_name}'")
            print(f"Existing documents in the collection: {self.collection.count()}")
        except Exception as e:
            print(f"Failed to initialize vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        if len(documents) != embeddings.shape[0]:
            raise ValueError("Number of documents and embeddings must match.")
        print(f"Adding {len(documents)} documents to the vector store...")

        ids = []
        metadatas = []
        document_texts = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            document_texts.append(doc.page_content)
            embeddings_list.append(embedding.tolist())  # Convert numpy array to list for storage

        # Add to the collection ONCE, after the loop finishes building all lists
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=document_texts
            )
            print(f"Successfully added {len(documents)} documents to the vector store.")
            print(f"Total documents in the collection after addition: {self.collection.count()}")
        except Exception as e:
            print(f"Failed to add documents to the vector store: {e}")
            raise


# Initialize the vector store (OUTSIDE the class, at top level)
vector_store = VectorStore()
vector_store

Vector store initialized. Collection: 'pdf_documents'
Existing documents in the collection: 0


In [44]:
chunks

[Document(metadata={'producer': 'macOS Version 12.5 (Build 21G72) Quartz PDFContext', 'creator': 'Pages', 'creationdate': '2022-08-06T19:34:55+00:00', 'moddate': '2022-08-07T01:05:26+05:30', 'title': 'lec11Notes', 'source': '../data\\lec11Notes.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'filename': 'lec11Notes.pdf', 'page_number': 1}, page_content="LEC-11: Normalisation \n1. Normalisation is a step towards DB optimisation.\n2. Functional Dependency (FD)\n1. It's a relationship between the primary key a ttribute (usually) of the relation to that of the other attribute of the\nrelation.\n2. X -> Y, the left side of FD is known as a Determinant, the right side of the production is known as a Dependent.\n3. Types of FD\n1. Trivial FD\n1. A → B has trivial functional dependency if B is a subset of A. A->A, B->B are also Trivial FD.\n2. Non-trivial FD\n1. A → B has a non-trivial functional dependency if B is not a subset of A. [A intersection B is NULL].\n4. Rules of FD (Armstrong

In [57]:
# ✅ Correct — keep the manager object and its output in separate variables
embedding_manager = EmbeddingManager()

texts = [chunk.page_content for chunk in chunks]
embeddings = embedding_manager.generate_embeddings(texts)   # store in `embeddings`, not `embedding_manager`

vector_store = VectorStore()
vector_store.add_documents(chunks, embeddings)

rag_retriever = RAGRetriever(vector_store, embedding_manager)  # now this is still the real object

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 391.85it/s]


Model loaded successfully. Embedding dimensions: 384
Generating embeddings for 115 texts...


Batches: 100%|██████████| 4/4 [00:29<00:00,  7.40s/it]


Embeddings generated. Shape: (115, 384)
Vector store initialized. Collection: 'pdf_documents'
Existing documents in the collection: 115
Adding 115 documents to the vector store...
Successfully added 115 documents to the vector store.
Total documents in the collection after addition: 230


RAG RETREIVAL PIPELINE


In [58]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vector_store,embedding_manager)

rag retreiver


In [60]:
rag_retriever = RAGRetriever(vector_store, embedding_manager)
rag_retriever.retrieve("what is the entity relationship model?")

Retrieving documents for query: 'what is the entity relationship model?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  4.40it/s]

Embeddings generated. Shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_47e6105f_70',
  'content': 'LEC-3: Entity-Relationship Model \n1. Data Model: Collection of conceptual tools for describing data, data relationships, data semantics, and consistency\nconstraints.\n2. ER Model\n1. It is a high level data model based on a perception of a real world that consists of a collection of basic objects, called\nentities and of relationships among these objects.\n2. Graphical representation of ER Model is ER diagram, which acts as a blueprint of DB.\n3. Entity: An Entity is a “thing” or “object” in the real world that is distinguishable from all other objects.\n1. It has physical existence.\n2. Each student in a college is an entity.\n3. Entity can be uniquely identified. (By a primary attribute, aka Primary Key)\n4. Strong Entity: Can be uniquely identified.\n5. Weak Entity: Can’t be uniquely identified., depends on some other strong entity.\n1. It doesn’t have suﬃcient attributes, to select a uniquely identi fiable attribute.',
  'metadata': {'page